# 04. Fit and similarity (Phase 4)

**What?** Three questions about how a player fits a club: can he play the slot the club needs
(role fit), does the club's style of play matter for how he will perform (style fit), and who
are the players most like him (similarity).

**Why?** The shortlist should not offer a striker for a full-back's slot, and the spec wanted style
and similarity to help rank candidates. Each idea has to earn its place on the data: a test that
can fail is set for every one, and a failure is reported, not hidden.

Inputs: the Phase 1 tables,
the Phase 2 contribution numbers, the Phase 3 market rows.

## Contents

- Step 1. Role fit: which slots can a player play, and what does switching cost?
- Step 2. Style fit: does the destination's way of playing change a player's output?
  - Steps 1 and 2, what we got
  - Step 2 in particular
- Step 3. Similarity: who are the players most like X?
  - Step 3. What we got
  - Step 3 check. The package reproduces the numbers, plus five examples


In [1]:
import json

import numpy as np
import pandas as pd

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 300)

from scout import config
from scout.data import understat
from scout.models import quantities
from scout.panel import player_match, team_season

LEAGUE_TO_COMP = {league: comp for comp, league in config.BIG5.items()}
rng = np.random.default_rng(0)

## Step 1. Role fit: which slots can a player play, and what does switching cost?

**What?** For every player-season, the share of his minutes in each of the six roles. Then the
players whose main role changed from one season to the next: how their output changed compared
with similar players who did not switch.

**Why?** A club looking for a full-back should only be shown players who can actually play there.
And if switching role usually costs output, the shortlist should price that in.

In [2]:
pm = player_match.build()
pm["competition_id"] = pm.league.map(LEAGUE_TO_COMP)
shots = understat.load("shots")
shots["competition_id"] = shots.league.map(LEAGUE_TO_COMP)
contrib = pd.DataFrame(json.load(open(config.MODELS / "phase2_contribution.json")))

# minutes by role per player-season (all roles, including sub minutes assigned to the season main role)
role_minutes = (
    pm.dropna(subset=["role"])
    .groupby(["competition_id", "season", "player_id", "role"])
    .minutes.sum()
    .unstack("role")
    .fillna(0)
)

total = role_minutes.sum(axis=1)
shares = role_minutes.div(total, axis=0)
regular = shares[total >= quantities.MIN_MINUTES]
main_share = regular.max(axis=1)

print(
    len(regular),
    "player-seasons ≥600 min | share of minutes in the main role — median",
    round(main_share.median(), 3),
    "| < 0.8 (two-role players):",
    f"{(main_share < 0.8).mean():.1%}",
    "| < 0.6:",
    f"{(main_share < 0.6).mean():.1%}",
)

second = regular.apply(
    lambda r: (
        r.sort_values(ascending=False).index[1]
        if r.sort_values(ascending=False).iloc[1] >= 0.2
        else None
    ),
    axis=1,
)

main = regular.idxmax(axis=1)
pairs_count = pd.crosstab(main[second.notna()], second[second.notna()])

print("\ncommon two-role pairs (main role × second role with ≥20% of minutes):")
print(pairs_count.to_string())

22168 player-seasons ≥600 min | share of minutes in the main role — median 1.0 | < 0.8 (two-role players): 18.6% | < 0.6: 4.3%



common two-role pairs (main role × second role with ≥20% of minutes):
col_0   CB   CM   FB   ST    W
row_0                         
CB       0  112  332    0    6
CM     106    0   84   12  581
FB     299   71    0    0  384
ST       0    7    1    0  491
W        5  488  230  564    0


In [3]:
# switchers: main role changes between consecutive seasons; outcome = next-season shrunk point relative to this season's
one = contrib.sort_values("minutes", ascending=False).drop_duplicates(["player_id", "season"])[
    ["player_id", "season", "role", "point", "minutes", "competition_id"]
]
one = one.merge(
    pd.DataFrame(json.load(open(config.MODELS / "phase3_market.json"))["rows"])[
        ["player_id", "season", "age"]
    ].drop_duplicates(["player_id", "season"]),
    on=["player_id", "season"],
    how="left",
)

nxt = one.assign(season=one.season - 1)[["player_id", "season", "role", "point"]].rename(
    columns={"role": "role_next", "point": "point_next"}
)

seq = one.merge(nxt, on=["player_id", "season"])
seq["switch"] = seq.role != seq.role_next
seq["delta"] = seq.point_next - seq.point

role_mean = one.groupby("role").point.mean()

seq["delta_rel"] = seq.delta / seq.role.map(
    role_mean
)  # change relative to the old role's typical output

print(
    f"consecutive-season pairs: {len(seq)} | switchers: {seq.switch.sum()} ({seq.switch.mean():.1%})"
)

seq["point_tercile"] = seq.groupby(["role", "season"]).point.transform(
    lambda x: pd.qcut(x.rank(method="first"), 3, labels=["low", "mid", "high"])
)
seq["age_band"] = pd.cut(seq.age, [15, 23, 27, 31, 45], labels=["≤23", "24-27", "28-31", "32+"])

keys = ["role", "season", "point_tercile", "age_band"]
control = seq[~seq.switch].groupby(keys, observed=True).delta.mean().rename("control_delta")
sw = seq[seq.switch].merge(control, on=keys, how="left").dropna(subset=["control_delta"])
sw["cost"] = (
    sw.delta - sw.control_delta
)  # negative = the switch cost output relative to matched non-switchers
sw["type"] = sw.role + "→" + sw.role_next

rows = []

for t, g in sw.groupby("type"):
    if len(g) < 15:
        continue

    boots = [
        g.cost.sample(len(g), replace=True, random_state=int(rng.integers(1e9))).mean()
        for _ in range(500)
    ]
    rows.append(
        {
            "switch": t,
            "n": len(g),
            "cost (per 90)": round(g.cost.mean(), 4),
            "p10": round(np.percentile(boots, 10), 4),
            "p90": round(np.percentile(boots, 90), 4),
            "cost as share of old role mean": round(g.cost.mean() / role_mean[g.role.iloc[0]], 3),
        }
    )

table = pd.DataFrame(rows).sort_values("n", ascending=False)
table["verdict"] = np.where(
    (table.p10 > 0) | (table.p90 < 0), "carries a cost/gain", "free (interval includes 0)"
)

print(table.to_string(index=False))

print(
    "\nall switches pooled: cost",
    round(sw.cost.mean(), 4),
    "| n",
    len(sw),
    "| share with a fall > 20% of the old role mean:",
    f"{(sw.cost < -0.2 * sw.role.map(role_mean)).mean():.1%}",
)

consecutive-season pairs: 12445 | switchers: 1470 (11.8%)


switch   n  cost (per 90)     p10     p90  cost as share of old role mean             verdict
  W→CM 230        -0.1419 -0.1486 -0.1348                          -0.366 carries a cost/gain
  W→ST 211         0.1098  0.0978  0.1210                           0.283 carries a cost/gain
  CM→W 194         0.1461  0.1371  0.1546                           0.940 carries a cost/gain
  ST→W 187        -0.0728 -0.0834 -0.0629                          -0.137 carries a cost/gain
 FB→CB 144        -0.0549 -0.0591 -0.0506                          -0.429 carries a cost/gain
  W→FB 125        -0.1493 -0.1591 -0.1400                          -0.385 carries a cost/gain
 CB→FB  92         0.0499  0.0427  0.0576                           0.795 carries a cost/gain
  FB→W  85         0.1451  0.1308  0.1598                           1.135 carries a cost/gain
 CM→CB  64        -0.0591 -0.0669 -0.0520                          -0.380 carries a cost/gain
 CB→CM  49         0.0363  0.0266  0.0469                   

**Problem with the table above.** It mostly measures how much each role scores, not how well the
player did: a winger who becomes a central midfielder "loses" 0.14 per 90 simply because
midfielders produce 0.16 where wingers produce 0.41. The fairer question is whether the player's
standing *within his new role* is worse than his standing in the old one, compared with players
who stayed put. That is the next cell (z-scores within role).

In [4]:
one["z"] = one.groupby(["role", "season"]).point.transform(lambda x: (x - x.mean()) / x.std())

nxt_z = one.assign(season=one.season - 1)[["player_id", "season", "role", "z"]].rename(
    columns={"role": "role_next", "z": "z_next"}
)

seqz = one.merge(nxt_z, on=["player_id", "season"])
seqz["switch"] = seqz.role != seqz.role_next
seqz["dz"] = seqz.z_next - seqz.z

seqz["z_tercile"] = seqz.groupby(["role", "season"]).z.transform(
    lambda x: pd.qcut(x.rank(method="first"), 3, labels=["low", "mid", "high"])
)
seqz["age_band"] = pd.cut(seqz.age, [15, 23, 27, 31, 45], labels=["≤23", "24-27", "28-31", "32+"])

keysz = ["role", "season", "z_tercile", "age_band"]
ctrl = seqz[~seqz.switch].groupby(keysz, observed=True).dz.mean().rename("control_dz")
swz = seqz[seqz.switch].merge(ctrl, on=keysz, how="left").dropna(subset=["control_dz"])
swz["cost_z"] = swz.dz - swz.control_dz
swz["type"] = swz.role + "→" + swz.role_next

rows = []

for t, g in swz.groupby("type"):
    if len(g) < 15:
        continue

    boots = [
        g.cost_z.sample(len(g), replace=True, random_state=int(rng.integers(1e9))).mean()
        for _ in range(500)
    ]
    rows.append(
        {
            "switch": t,
            "n": len(g),
            "cost (z within role)": round(g.cost_z.mean(), 3),
            "p10": round(np.percentile(boots, 10), 3),
            "p90": round(np.percentile(boots, 90), 3),
        }
    )

tz = pd.DataFrame(rows).sort_values("n", ascending=False)
tz["verdict"] = np.where(
    (tz.p10 > 0) | (tz.p90 < 0), "carries a cost/gain", "free (interval includes 0)"
)

print(tz.to_string(index=False))

print(
    "\nall switches pooled: cost",
    round(swz.cost_z.mean(), 3),
    "z | n",
    len(swz),
    "| non-switchers' mean dz (regression to the mean):",
    round(seqz[~seqz.switch].dz.mean(), 3),
)

switch   n  cost (z within role)    p10    p90                    verdict
  W→CM 230                 1.042  0.968  1.118        carries a cost/gain
  W→ST 211                -0.269 -0.355 -0.187        carries a cost/gain
  CM→W 194                -0.991 -1.074 -0.909        carries a cost/gain
  ST→W 187                 0.550  0.467  0.642        carries a cost/gain
 FB→CB 144                 0.169  0.048  0.276        carries a cost/gain
  W→FB 125                 1.306  1.205  1.424        carries a cost/gain
 CB→FB  92                -0.256 -0.398 -0.117        carries a cost/gain
  FB→W  85                -1.144 -1.270 -1.003        carries a cost/gain
 CM→CB  64                 0.360  0.175  0.529        carries a cost/gain
 CB→CM  49                -0.661 -0.810 -0.507        carries a cost/gain
 CM→FB  26                 0.060 -0.171  0.302 free (interval includes 0)
 FB→CM  21                -0.213 -0.366 -0.064        carries a cost/gain

all switches pooled: cost 0.107 z | n

## Step 2. Style fit: does the destination's way of playing change a player's output?

**What?** For 2,604 players who moved between Big-5 clubs, we compare their output after the
move with their output before, against two measures of the destination: (a) how different its
style is from the club they left, (b) how the player's own profile interacts with the new club's
style. Players are compared only with similar players (same role, season, quality and age).

**Why?** The spec wanted style fit in the ranking. But clubs choose whom they buy, so a plain
"players at high-pressing clubs do well" comparison would be misleading. The rule from the
spec: if no effect can be told from zero, style fit stays out of the ranking.

In [5]:
import statsmodels.formula.api as smf

ts = team_season.build()
ts["competition_id"] = ts.league.map(LEAGUE_TO_COMP)
STYLE = team_season.STYLE
resid = ts.copy()

for comp_col in [c for c in STYLE if c != "expected_points_for"]:
    fit = smf.ols(
        f"{comp_col} ~ expected_points_for + C(competition_id) + C(season)", data=ts
    ).fit()
    resid[comp_col] = fit.resid / fit.resid.std()

style = resid.set_index(["competition_id", "season", "team_id"])[
    [c for c in STYLE if c != "expected_points_for"]
]

STYLE_COLS = list(style.columns)

# movers at club level from the per-match table (main club per season)
club_season = (
    pm.dropna(subset=["role"])
    .groupby(["competition_id", "season", "player_id", "team_id"])
    .minutes.sum()
    .reset_index()
)
club_season = club_season.sort_values("minutes", ascending=False).drop_duplicates(
    ["player_id", "season"]
)

mv = one.merge(club_season[["player_id", "season", "team_id"]], on=["player_id", "season"])

nxt_mv = mv.assign(season=mv.season - 1)[
    ["player_id", "season", "team_id", "point", "role", "competition_id"]
].rename(
    columns={
        "team_id": "team_next",
        "point": "point_next",
        "role": "role_next",
        "competition_id": "comp_next",
    }
)

mv = mv.merge(nxt_mv, on=["player_id", "season"])
mv = mv[(mv.team_id != mv.team_next) & (mv.role == mv.role_next)].copy()
mv["log_ratio"] = np.log(mv.point_next.clip(lower=0.02) / mv.point.clip(lower=0.02))
mv["age_band"] = pd.cut(mv.age, [15, 23, 27, 31, 45], labels=["≤23", "24-27", "28-31", "32+"])

origin = style.reindex(
    pd.MultiIndex.from_frame(mv[["competition_id", "season", "team_id"]])
).to_numpy()

dest_index = pd.MultiIndex.from_frame(
    mv[["comp_next", "season", "team_next"]].assign(season=mv.season + 1)
)

dest = style.reindex(dest_index).to_numpy()
mv["style_distance"] = np.sqrt(np.nansum((dest - origin) ** 2, axis=1))

for i, c in enumerate(STYLE_COLS):
    mv[f"dest_{c}"] = dest[:, i]

mv["z"] = mv.groupby(["role", "season"]).point.transform(lambda x: (x - x.mean()) / x.std())
mv = mv.dropna(subset=["style_distance"] + [f"dest_{c}" for c in STYLE_COLS])

print(
    len(mv),
    "movers with both styles | mean log ratio (regression to the mean + age):",
    round(mv.log_ratio.mean(), 3),
)

base = smf.ols("log_ratio ~ C(role) + C(age_band) + z", data=mv).fit()
mv["net"] = base.resid

print(
    "style distance: mean",
    round(mv.style_distance.mean(), 2),
    "| effect of +1 sd of distance on the net log ratio:",
)

fa = smf.ols("net ~ style_distance", data=mv).fit()

print(
    f"  (a) distance: {fa.params['style_distance'] * mv.style_distance.std():+.4f} per sd, 80% CI [{(fa.params['style_distance'] - 1.2816 * fa.bse['style_distance']) * mv.style_distance.std():+.4f}, {(fa.params['style_distance'] + 1.2816 * fa.bse['style_distance']) * mv.style_distance.std():+.4f}], n={len(mv)}"
)
print("  (b) player output z × destination component (effect per sd of the interaction, 80% CI):")

for c in STYLE_COLS:
    mv["inter"] = mv.z * mv[f"dest_{c}"]
    fb = smf.ols("net ~ inter + " + f"dest_{c}", data=mv).fit()
    lo, hi = (
        fb.params["inter"] - 1.2816 * fb.bse["inter"],
        fb.params["inter"] + 1.2816 * fb.bse["inter"],
    )
    sd = mv.inter.std()

    print(
        f"     {c:28s} {fb.params['inter'] * sd:+.4f} [{lo * sd:+.4f}, {hi * sd:+.4f}]{'  <- excludes 0' if lo > 0 or hi < 0 else ''}"
    )

2604 movers with both styles | mean log ratio (regression to the mean + age): -0.005
style distance: mean 3.08 | effect of +1 sd of distance on the net log ratio:
  (a) distance: +0.0088 per sd, 80% CI [-0.0025, +0.0202], n=2604
  (b) player output z × destination component (effect per sd of the interaction, 80% CI):
     np_xg_for                    -0.0051 [-0.0165, +0.0063]
     np_xg_against                -0.0053 [-0.0167, +0.0060]
     ppda_for                     +0.0088 [-0.0026, +0.0201]
     ppda_against                 +0.0233 [+0.0118, +0.0348]  <- excludes 0
     deep_completions_for         +0.0062 [-0.0052, +0.0177]
     deep_completions_against     -0.0104 [-0.0218, +0.0009]


In [6]:
# held-out gain: does any exposure improve the prediction of the post-move ratio beyond role, age and z?
def lfo(formula):
    errs = []

    for s in range(2018, 2025):
        tr, te = mv[mv.season < s], mv[mv.season == s]

        if len(te) < 20:
            continue

        pred = smf.ols(formula, data=tr).fit().predict(te)
        errs.append((te.log_ratio - pred).abs())

    e = pd.concat(errs)
    return round(e.mean(), 4), len(e)


print("held-out MAE of the post-move log ratio:")
print("  base (role, age band, z):", lfo("log_ratio ~ C(role) + C(age_band) + z"))
print("  + style distance:", lfo("log_ratio ~ C(role) + C(age_band) + z + style_distance"))
print(
    "  + all destination components:",
    lfo("log_ratio ~ C(role) + C(age_band) + z + " + " + ".join(f"dest_{c}" for c in STYLE_COLS)),
)
print(
    "  + z × all destination components:",
    lfo(
        "log_ratio ~ C(role) + C(age_band) + z + "
        + " + ".join(f"z:dest_{c} + dest_{c}" for c in STYLE_COLS)
    ),
)

# matched high/low exposure comparison on the distance
mv["hi_dist"] = mv.groupby(["role", "season"]).style_distance.transform(lambda x: x > x.median())

m = (
    mv.groupby(["role", "season", "age_band", "hi_dist"], observed=True)
    .net.mean()
    .unstack("hi_dist")
    .dropna()
)

diff = m[True] - m[False]

boots = [
    diff.sample(len(diff), replace=True, random_state=int(rng.integers(1e9))).mean()
    for _ in range(500)
]

print(
    f"\nmatched cells (role × season × age band): high-distance movers minus low-distance movers, net log ratio = {diff.mean():+.4f}, 80% CI [{np.percentile(boots, 10):+.4f}, {np.percentile(boots, 90):+.4f}] over {len(diff)} cells"
)

held-out MAE of the post-move log ratio:
  base (role, age band, z): (np.float64(0.3187), 1571)
  + style distance: (np.float64(0.3185), 1571)


  + all destination components: (np.float64(0.3172), 1571)


  + z × all destination components: (np.float64(0.318), 1571)

matched cells (role × season × age band): high-distance movers minus low-distance movers, net log ratio = +0.0058, 80% CI [-0.0227, +0.0347] over 203 cells


### Steps 1 and 2. What we got

**Slot minutes.** Of 22,168 player-seasons with 600+ minutes, the typical player plays all his
minutes in one role; 18.6% play a second role for at least a fifth of their minutes (4.3% have
no role above 60%). The usual pairs: winger ↔ central midfielder (581 / 488 players),
winger ↔ striker (491 / 564), full-back ↔ centre-back (332 / 299), full-back ↔ winger
(384 / 230). Strikers and centre-backs never overlap.

**Switch cost cannot be measured from output.** 11.8% of consecutive player-seasons change main
role. On the raw scale the "cost" is just the roles' different scoring levels. Standardised
within role it flips sign and gets bigger: a winger moving to midfield "gains" +1.04 standard
deviations, a midfielder moving to the wing "loses" −0.99. That is not fit, a winger carries his
attacking numbers into a role that does not produce them, and the reverse, plus selection
(wingers become strikers because they score). Neither table is an honest cost.

**Decision.** Role fit = *eligibility*: a player can be asked to play his main role and any role he
has played in for at least 20% of his recent minutes. Switches between eligible roles are treated
as free (the spec's rule for costs that cannot be shown), other roles are excluded from a search.
The switch tables stay as descriptive evidence. Ported: `scout.models.fit` (`role_shares`,
`eligible_slots`).

### Step 2 in particular

**Style fit fails its test and stays out of the ranking.** On 2,604 movers with a style vector at
both clubs (the seven Phase 1 style components, with the club's strength removed), neither
measure moves output after the move once role, age and quality are taken into account: a one
standard deviation bigger style difference changes output by +0.9% (80% interval −0.3% to
+2.0%, includes zero); matched groups of high- vs low-difference movers differ by +0.6%
(interval −2.3% to +3.5%); of six profile × style interactions one clears zero at the 80%
level, which is what chance gives; and no measure improves the held-out prediction of the
post-move change (error 0.3187 → 0.3172 at best).

**Decision.** Style fit is shown as a description next to a shortlist ("this club plays quite
differently from his current one"), never as a term in the ranking. This is the second piece of
scouting intuition the data declined (after the big-game player in Phase 2); the writeup shows
both.

## Step 3. Similarity: who are the players most like X?

**What?** A distance between players of the same role, built from what a player does per 90
(the Phase 2 quantities), where he shoots from (average distance, share inside the box, share
from the left), and which roles he plays. Two versions: plain standardised distance, and a
version that weights each feature by how stable it is year to year.

**Why?** "Players like X" is the natural way to start a search for a replacement. A distance is
only useful if similar players actually behave similarly, so two tests: do a player's ten nearest
neighbours tell us anything about his *next* season beyond his own history, and do the
neighbours stay the same from one season to the next?

In [7]:
from scout.models import fit as fit_model

per90 = quantities.season_role_per90(pm, shots)
per90 = per90[per90.minutes >= quantities.MIN_MINUTES].copy()
Q = quantities.UNDERSTAT  # npxg, xa, key_passes, shots, xg_chain, xg_buildup

# shot locations: Understat pitch coordinates, x toward the goal at 1, y across the pitch
sh = shots[shots.result != "Own Goal"].copy()
sh["dist"] = np.sqrt(((1 - sh.location_x) * 105) ** 2 + ((sh.location_y - 0.5) * 68) ** 2)
sh["in_box"] = (sh.location_x >= 0.843) & (sh.location_y.between(0.211, 0.789))
sh["left"] = sh.location_y < 0.5

loc = (
    sh.groupby(["competition_id", "season", "player_id"])
    .agg(
        shot_dist=("dist", "mean"),
        box_share=("in_box", "mean"),
        left_share=("left", "mean"),
        n_shots=("dist", "size"),
    )
    .reset_index()
)

shares = fit_model.role_shares(pm)

feat = per90.merge(loc, on=["competition_id", "season", "player_id"], how="left").merge(
    shares, on=["competition_id", "season", "player_id"], how="left"
)

for c in ["shot_dist", "box_share", "left_share"]:
    feat[c] = feat[c].where(
        feat.n_shots >= 10
    )  # fewer than 10 shots says nothing about where a player shoots

LOC = ["shot_dist", "box_share", "left_share"]
SH = [f"{r}" for r in fit_model.ROLES]
FEATURES = Q + LOC + SH
feat = feat.sort_values("minutes", ascending=False).drop_duplicates(["player_id", "season", "role"])

print(
    len(feat),
    "player-season-roles | shot-location coverage:",
    f"{feat.shot_dist.notna().mean():.1%}",
)

# standardise within role; missing location -> role mean (0 after standardising)
Z = feat.copy()


def zscore(x):
    x = x.astype(float)
    sd = x.std()
    return (x - x.mean()) / (sd if sd > 0 else 1.0)


for c in FEATURES:
    Z[c] = feat.groupby("role")[c].transform(zscore).astype(float).fillna(0.0)

# year-to-year signal per feature within role, for the weights of candidate (b)
nxt = Z.assign(season=Z.season - 1)[["player_id", "season", "role"] + FEATURES].rename(
    columns={c: f"{c}_next" for c in FEATURES}
)

pz = Z.merge(nxt, on=["player_id", "season", "role"])
signal = pd.Series({c: pz[c].corr(pz[f"{c}_next"]) for c in FEATURES}).fillna(0).clip(lower=0)

print("year-to-year r per feature (weights for candidate b):")
print(signal.round(2).to_string())

22996 player-season-roles | shot-location coverage: 70.2%
year-to-year r per feature (weights for candidate b):
npxg          0.49
xa            0.45
key_passes    0.58
shots         0.59
xg_chain      0.66
xg_buildup    0.62
shot_dist     0.63
box_share     0.59
left_share    0.45
GK            0.00
CB            0.46
FB            0.49
CM            0.47
W             0.43
ST            0.30


/Users/mihailandreev/football-player-scouting/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]


In [8]:
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

Z["expected_output"] = Z.npxg_raw if "npxg_raw" in Z else feat.npxg + feat.xa
Z["out_next"] = Z.merge(
    Z.assign(season=Z.season - 1)[["player_id", "season", "role", "expected_output"]].rename(
        columns={"expected_output": "out_next"}
    ),
    on=["player_id", "season", "role"],
    how="left",
).out_next.values


def embed(frame, candidate):
    X = frame[FEATURES].to_numpy(dtype=float)

    if candidate == "a":
        return X

    Xw = X * np.sqrt(signal.to_numpy())
    return PCA(whiten=True, random_state=0).fit_transform(Xw)


def validity(candidate, k=10):
    rows = []

    for role, g in Z.groupby("role"):
        for s in range(2016, 2025):
            base = g[g.season == s].dropna(subset=["out_next"]).copy()

            if len(base) < 40:
                continue

            E = embed(base, candidate)
            nn = NearestNeighbors(n_neighbors=k + 1).fit(E)
            _, idx = nn.kneighbors(E)
            neigh_next = base.out_next.to_numpy()[idx[:, 1:]].mean(axis=1)
            own = base.expected_output.to_numpy()
            target = base.out_next.to_numpy()
            X1 = np.c_[np.ones(len(base)), own]
            X2 = np.c_[np.ones(len(base)), own, neigh_next]
            r1 = np.corrcoef(X1 @ np.linalg.lstsq(X1, target, rcond=None)[0], target)[0, 1]
            r2 = np.corrcoef(X2 @ np.linalg.lstsq(X2, target, rcond=None)[0], target)[0, 1]
            rows.append(
                {"role": role, "season": s, "n": len(base), "own r": r1, "own + neighbours r": r2}
            )

    return pd.DataFrame(rows)


def stability(candidate, k=10):
    overlaps = []

    for role, g in Z.groupby("role"):
        for s in range(2016, 2024):
            a, b = g[g.season == s], g[g.season == s + 1]
            common = np.intersect1d(a.player_id, b.player_id)

            if len(common) < 40:
                continue

            a, b = (
                a.set_index("player_id").loc[common].reset_index(),
                b.set_index("player_id").loc[common].reset_index(),
            )
            Ea, Eb = embed(a, candidate), embed(b, candidate)
            ia = NearestNeighbors(n_neighbors=k + 1).fit(Ea).kneighbors(Ea)[1][:, 1:]
            ib = NearestNeighbors(n_neighbors=k + 1).fit(Eb).kneighbors(Eb)[1][:, 1:]
            overlaps.append(
                np.mean([len(set(x) & set(y)) / k for x, y in zip(ia, ib, strict=True)])
            )

    return float(np.mean(overlaps))


for cand, label in [
    ("a", "(a) standardised Euclidean"),
    ("b", "(b) signal-weighted, PCA-whitened"),
]:
    v = validity(cand)

    print(
        f"{label}: own r {np.average(v['own r'], weights=v.n):.3f} → own + neighbours r {np.average(v['own + neighbours r'], weights=v.n):.3f} | top-10 overlap year to year {stability(cand):.3f}"
    )

(a) standardised Euclidean: own r 0.512 → own + neighbours r 0.525 | top-10 overlap year to year 0.122


(b) signal-weighted, PCA-whitened: own r 0.512 → own + neighbours r 0.522 | top-10 overlap year to year 0.103


### Step 3. What we got

**Chosen: the plain standardised distance within role** over 15 features (six output quantities,
three shot-location summaries, only for players with 10+ shots, 70% of them, and six role
shares). Ported: `scout.models.similarity`.

The neighbourhood carries real but small information: a player's ten nearest neighbours' next
season predicts his own next season a little beyond his own history (r 0.512 → 0.525, held out).
Neighbour sets overlap 12% from one season to the next, five times what chance would give with
about 400 players per role and season, and still low, because one season's profile is noisy.
The weighted, PCA-whitened version was no better (0.522, 10% overlap) and is dropped.

**What this means.** "Players like X" is a browsing view with a validated but small predictive
value. The ranking itself rests on contribution, surplus and price, never on similarity alone.

### Step 3 check. The package reproduces the numbers, plus five examples

**What?** The same validity test through `scout.models.similarity`, and the nearest neighbours of
the five highest-output attackers and midfielders of 2024-25, by name, as a sanity check.

In [9]:
from scout.models import similarity as similarity_model

prof = similarity_model.profile(pm, shots)

print(len(prof), "profiles (above: 22,996)")

prof["expected_output"] = (
    prof.npxg + prof.xa
)  # per-90 columns are standardised in prof: use the raw per90 table for output

raw_out = (
    per90.sort_values("minutes", ascending=False)
    .drop_duplicates(["player_id", "season", "role"])
    .set_index(["player_id", "season", "role"])[["npxg", "xa"]]
    .sum(axis=1)
)  # a cross-league mover has two rows

prof["expected_output"] = raw_out.reindex(
    pd.MultiIndex.from_frame(prof[["player_id", "season", "role"]])
).to_numpy()
prof["out_next"] = (
    raw_out.rename("x")
    .reset_index()
    .assign(season=lambda d: d.season - 1)
    .set_index(["player_id", "season", "role"])
    .x.reindex(pd.MultiIndex.from_frame(prof[["player_id", "season", "role"]]))
    .to_numpy()
)

nb_pkg = similarity_model.neighbours(prof)
lookup = prof.set_index(["role", "season", "player_id"]).out_next
rows = []

for (role, season), g in nb_pkg.groupby(["role", "season"]):
    base = (
        prof[(prof.role == role) & (prof.season == season)]
        .dropna(subset=["out_next"])
        .set_index("player_id")
    )

    if len(base) < 40 or season < 2016 or season > 2024:
        continue

    g = g[g.player_id.isin(base.index)]
    neigh = np.array(
        [np.nanmean([lookup.get((role, season, n), np.nan) for n in ns]) for ns in g.neighbours]
    )
    own = base.loc[g.player_id, "expected_output"].to_numpy()
    target = base.loc[g.player_id, "out_next"].to_numpy()
    ok = ~np.isnan(neigh)
    X1 = np.c_[np.ones(ok.sum()), own[ok]]
    X2 = np.c_[np.ones(ok.sum()), own[ok], neigh[ok]]
    rows.append(
        {
            "n": ok.sum(),
            "own": np.corrcoef(X1 @ np.linalg.lstsq(X1, target[ok], rcond=None)[0], target[ok])[
                0, 1
            ],
            "both": np.corrcoef(X2 @ np.linalg.lstsq(X2, target[ok], rcond=None)[0], target[ok])[
                0, 1
            ],
        }
    )

v = pd.DataFrame(rows)

print(
    f"package: own r {np.average(v.own, weights=v.n):.3f} → own + neighbours r {np.average(v.both, weights=v.n):.3f} (above: 0.512 → 0.525)"
)

# examples: the five highest-output 2024-25 attackers and midfielders, with their nearest neighbours by name
names = understat.load("player_season").drop_duplicates("player_id").set_index("player_id").player
latest = prof[prof.season == 2024]

top5 = (
    latest[latest.role.isin(["W", "ST", "CM"])]
    .sort_values("expected_output", ascending=False)
    .drop_duplicates("player_id")
    .head(5)
)

for _, r in top5.iterrows():
    ns = nb_pkg[
        (nb_pkg.role == r.role) & (nb_pkg.season == 2024) & (nb_pkg.player_id == r.player_id)
    ].neighbours.iloc[0][:5]

    print(
        f"{names.get(r.player_id, r.player_id)} ({r.role}, {r.expected_output:.2f}/90): "
        + ", ".join(str(names.get(n, n)) for n in ns)
    )

22996 profiles (above: 22,996)


/var/folders/s1/ydqc1kqd1_l_3stqdlcsdygm0000gn/T/ipykernel_86491/2173344056.py:17: RuntimeWarning: Mean of empty slice
  neigh = np.array([np.nanmean([lookup.get((role, season, n), np.nan) for n in ns]) for ns in g.neighbours])
/var/folders/s1/ydqc1kqd1_l_3stqdlcsdygm0000gn/T/ipykernel_86491/2173344056.py:17: RuntimeWarning: Mean of empty slice
  neigh = np.array([np.nanmean([lookup.get((role, season, n), np.nan) for n in ns]) for ns in g.neighbours])
/var/folders/s1/ydqc1kqd1_l_3stqdlcsdygm0000gn/T/ipykernel_86491/2173344056.py:17: RuntimeWarning: Mean of empty slice
  neigh = np.array([np.nanmean([lookup.get((role, season, n), np.nan) for n in ns]) for ns in g.neighbours])


package: own r 0.512 → own + neighbours r 0.524 (above: 0.512 → 0.525)
Gonçalo Ramos (ST, 1.69/90): Harry Kane, Mika Biereth, Alexander Sørloth, Mateo Retegui, Ferrán Torres
Ousmane Dembélé (ST, 1.51/90): Ferrán Torres, Amine Gouiri, Georges Mikautadze, Ángel Correa, Hugo Ekitike
Ferrán Torres (ST, 1.49/90): Ousmane Dembélé, Kylian Mbappe-Lottin, Georges Mikautadze, Hugo Ekitike, Amine Gouiri
Alexander Sørloth (ST, 1.28/90): Robert Lewandowski, Diogo Jota, Ermedin Demirovic, Victor Boniface, Serhou Guirassy
Mika Biereth (ST, 1.22/90): Breel Embolo, Nick Woltemade, Georges Mikautadze, Hugo Ekitike, Mateo Retegui


/var/folders/s1/ydqc1kqd1_l_3stqdlcsdygm0000gn/T/ipykernel_86491/2173344056.py:17: RuntimeWarning: Mean of empty slice
  neigh = np.array([np.nanmean([lookup.get((role, season, n), np.nan) for n in ns]) for ns in g.neighbours])
/var/folders/s1/ydqc1kqd1_l_3stqdlcsdygm0000gn/T/ipykernel_86491/2173344056.py:17: RuntimeWarning: Mean of empty slice
  neigh = np.array([np.nanmean([lookup.get((role, season, n), np.nan) for n in ns]) for ns in g.neighbours])
